# NOTEBOOK 04 – TRÚC CHI | CHƯƠNG 4: MÔ HÌNH & THỰC NGHIỆM

**Phụ trách:** 4.1–4.5 · Issue **#5** · Hạn báo cáo/Validation: 22/11/2026; bàn giao dự đoán Test cho Vĩnh Chi trước 24/11.

**CHỈ HUẤN LUYỆN:** Dummy Classifier. **Tiếp nhận RF** từ Ngọc Lan và **MLP** từ Tấn Lên (TV1), không fit lại cả RF/MLP.

**Đầu vào** từ Lam Linh: `data/processed/orders_model_ready.csv`. **Bàn giao từ Tấn Lên:** `models/tan_len_mlp.joblib`, `mlp_val_predictions.csv`, `mlp_metadata.json`.

**Bàn giao từ Ngọc Lan:** `models/ngoc_lan_random_forest.joblib`, `data/processed/rf_val_predictions.csv`, `rf_metadata.json` dùng cùng split.

**Đầu ra**: `test_predictions.csv`, `experiment_metadata.json`, các model `joblib` (không push), bảng tham số và ngưỡng trên Validation.

[Lý thuyết Ch4](../docs/theory/04_truc_chi_ch4.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

> Chạy **chỉ sau khi Ch3 bàn giao** và audit. Không đưa thông tin tương lai vào X. Không báo kết quả trước khi chạy; các cell ở đây là code gợi ý chuẩn hóa để hoàn thiện.

In [13]:
from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

Project root: D:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02


### Kiểm tra dữ liệu đầu vào trước thực nghiệm

**Mục đích:** Kiểm tra sự hiện diện của 9 file CSV gốc thuộc bộ dữ liệu Olist, bảo đảm dữ liệu nguồn đã được chuẩn bị trước khi tiếp nhận dữ liệu xử lý từ Chương 3 và thực hiện chia Train/Validation/Test.

**Thực hiện:** Xác định đường dẫn `data/raw`, thống kê các file CSV, kiểm tra số lượng file và số cột của từng tập dữ liệu.

**Kết quả mong đợi:** Tìm thấy đầy đủ 9 file CSV gốc. Việc kiểm tra này chưa thay thế bước xác nhận dữ liệu đã xử lý và kiểm tra Data Leakage.

In [14]:

# KIEM TRA 9 FILE CSV GOC OLIST

# 1. Xac dinh thu muc goc du an
ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "raw").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError("Khong tim thay thu muc data/raw")

RAW_DIR = ROOT / "data" / "raw"

# 2. Lay danh sach file CSV
csv_files = sorted(RAW_DIR.glob("*.csv"))

print("===== KIEM TRA DU LIEU GOC OLIST =====")
print("So file CSV:", len(csv_files))

# 3. Kiem tra so luong file
assert len(csv_files) == 9, "Chua du 9 file CSV!"

# 4. Kiem tra ten file va so cot
for file in csv_files:
    columns = pd.read_csv(file, nrows=0).columns
    print(f"{file.name}: {len(columns)} cot")

print("\nPASS: Da tim thay du 9 file CSV goc.")

===== KIEM TRA DU LIEU GOC OLIST =====
So file CSV: 9
olist_customers_dataset.csv: 5 cot
olist_geolocation_dataset.csv: 5 cot
olist_order_items_dataset.csv: 7 cot
olist_order_payments_dataset.csv: 5 cot
olist_order_reviews_dataset.csv: 7 cot
olist_orders_dataset.csv: 8 cot
olist_products_dataset.csv: 9 cot
olist_sellers_dataset.csv: 4 cot
product_category_name_translation.csv: 2 cot

PASS: Da tim thay du 9 file CSV goc.


### Kiểm tra dữ liệu bàn giao từ Chương 3

**Mục đích:** Xác nhận sự hiện diện của các file dữ liệu đã xử lý do Lam Linh bàn giao trước khi thực hiện chia Train/Validation/Test.

**Nội dung kiểm tra:**
- File dữ liệu cấp đơn hàng `orders_model_ready.csv`.
- File đầu vào mô hình `orders_baseline_input.csv`.
- File thông tin chia dữ liệu `orders_split_metadata.csv`.
- Danh sách đặc trưng hợp lệ `feature_whitelist.json`.
- File `data_manifest.json` phục vụ kiểm tra thông tin và tính toàn vẹn dữ liệu.

**Kết quả mong đợi:** Xác định đầy đủ các file bàn giao để tiếp tục kiểm tra kích thước dữ liệu, nhãn, đặc trưng và mã SHA256.

In [15]:

# KIEM TRA CAC FILE DU LIEU BAN GIAO

PROCESSED = ROOT / "data" / "processed"

required_files = [
    "orders_model_ready.csv",
    "orders_baseline_input.csv",
    "orders_split_metadata.csv",
    "feature_whitelist.json",
    "data_manifest.json"
]

print("===== KIEM TRA DU LIEU BAN GIAO =====")

found = []
missing = []

for name in required_files:
    # Tim file trong thu muc data
    matches = list((ROOT / "data").rglob(name))

    if matches:
        found.append(name)
        print(f"PASS: {name}")
        print(f"Duong dan: {matches[0]}")
    else:
        missing.append(name)
        print(f"MISSING: {name}")

print("\n===== TONG KET =====")
print("So file tim thay:", len(found))
print("So file con thieu:", len(missing))

if not missing:
    print("PASS: Da tim thay du 5 file ban giao.")
else:
    print("Can tao hoac nhan cac file con thieu.")

===== KIEM TRA DU LIEU BAN GIAO =====
PASS: orders_model_ready.csv
Duong dan: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\data\processed\orders_model_ready.csv
PASS: orders_baseline_input.csv
Duong dan: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\data\processed\orders_baseline_input.csv
PASS: orders_split_metadata.csv
Duong dan: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\data\processed\orders_split_metadata.csv
PASS: feature_whitelist.json
Duong dan: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\data\processed\feature_whitelist.json
PASS: data_manifest.json
Duong dan: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\data\processed\data_manifest.json

===== TONG KET =====
So file tim thay: 5
So file con thieu: 0
PASS: Da tim thay du 5 file ban giao.


### Kiểm tra kích thước và tính hợp lệ của dữ liệu Model-Ready

**Mục đích:** Xác nhận dữ liệu sau xử lý từ Chương 3 đáp ứng yêu cầu bàn giao trước khi thực hiện huấn luyện và đánh giá mô hình.

**Nội dung kiểm tra:**
- Kiểm tra số dòng và số cột của tập dữ liệu `orders_model_ready.csv`.
- Xác minh mỗi `order_id` là duy nhất.
- Kiểm tra nhãn `is_late` không chứa giá trị thiếu và chỉ nhận hai giá trị 0 hoặc 1.

**Kết quả mong đợi:** Dữ liệu có 96.456 dòng, 37 cột, không trùng `order_id` và nhãn phân loại hợp lệ.

In [16]:

# KIEM TRA DU LIEU MODEL-READY

# Tim file du lieu da xu ly
matches = list((ROOT / "data").rglob("orders_model_ready.csv"))

if not matches:
    raise FileNotFoundError("Chua tim thay orders_model_ready.csv")

model_ready_path = matches[0]
df = pd.read_csv(model_ready_path)

print("===== KIEM TRA MODEL-READY =====")

# 1. Kiem tra kich thuoc
print("So dong:", df.shape[0])
print("So cot:", df.shape[1])

assert df.shape == (96456, 37), "Sai kich thuoc du lieu!"

# 2. Kiem tra order_id
assert "order_id" in df.columns, "Thieu order_id"

duplicate_count = df["order_id"].duplicated().sum()
print("So order_id trung:", duplicate_count)

assert duplicate_count == 0, "Co order_id bi trung!"
assert df["order_id"].notna().all(), "Co order_id bi thieu!"

# 3. Kiem tra nhan is_late
assert "is_late" in df.columns, "Thieu nhan is_late"

y = pd.to_numeric(df["is_late"], errors="coerce")

print("So nhan bi thieu:", y.isna().sum())
print("Phan bo nhan:")
print(y.value_counts())

assert y.notna().all(), "Nhan co gia tri thieu!"
assert set(y.unique()) == {0, 1}, "Nhan khong hop le!"

print("\nPASS: Du lieu model-ready hop le.")


===== KIEM TRA MODEL-READY =====
So dong: 96456
So cot: 37
So order_id trung: 0
So nhan bi thieu: 0
Phan bo nhan:
is_late
0    89922
1     6534
Name: count, dtype: int64

PASS: Du lieu model-ready hop le.


### Kiểm tra danh sách đặc trưng đầu vào (Feature Whitelist)

**Mục đích:** Xác nhận các đặc trưng được sử dụng trong mô hình phù hợp với danh sách Feature Whitelist đã được phê duyệt tại Chương 3.

**Nội dung kiểm tra:**
- Đọc danh sách đặc trưng từ `feature_whitelist.json`.
- Kiểm tra các cột trong `orders_baseline_input.csv`.
- Đối chiếu đặc trưng đầu vào với danh sách được phép sử dụng.
- Phát hiện các đặc trưng không hợp lệ có nguy cơ gây Data Leakage.

**Kết quả mong đợi:** Có đúng 5 đặc trưng đầu vào hợp lệ, phục vụ thống nhất cho Dummy Classifier, Random Forest và MLP.

In [18]:

# KIEM TRA FEATURE WHITELIST

# Tim hai file du lieu ban giao
data_dir = ROOT / "data"

whitelist_path = next(
    data_dir.rglob("feature_whitelist.json"), None
)

baseline_path = next(
    data_dir.rglob("orders_baseline_input.csv"), None
)

if whitelist_path is None or baseline_path is None:
    raise FileNotFoundError("Thieu file whitelist hoac baseline")

# Doc Feature Whitelist
with open(whitelist_path, "r", encoding="utf-8") as f:
    whitelist = json.load(f)

# Doc du lieu baseline
baseline = pd.read_csv(baseline_path)

print("===== KIEM TRA FEATURE WHITELIST =====")

print("\nNoi dung feature_whitelist.json:")
print(json.dumps(whitelist, indent=2, ensure_ascii=False))

print("\nKich thuoc baseline:", baseline.shape)

print("\nDanh sach cot trong baseline:")
for col in baseline.columns:
    print("-", col)


===== KIEM TRA FEATURE WHITELIST =====

Noi dung feature_whitelist.json:
[
  "purchase_month_number",
  "purchase_weekday",
  "purchase_hour",
  "approval_delay_hours",
  "customer_state"
]

Kich thuoc baseline: (96456, 7)

Danh sach cot trong baseline:
- order_id
- is_late
- purchase_month_number
- purchase_weekday
- purchase_hour
- approval_delay_hours
- customer_state


### Xác nhận Feature Whitelist và kiểm soát Data Leakage

**Mục đích:** Kiểm tra danh sách đặc trưng đầu vào đã được phê duyệt, đảm bảo dữ liệu baseline không chứa các biến hậu giao có nguy cơ gây Data Leakage.

**Phương pháp:** Sử dụng các câu lệnh `assert` để xác nhận số lượng đặc trưng, đối chiếu các cột dữ liệu với Feature Whitelist và kiểm tra sự xuất hiện của các Feature bị cấm.

**Kết quả mong đợi:** Dữ liệu có đúng 5 Feature đầu vào hợp lệ và không xuất hiện Feature bị cấm trực tiếp.

In [19]:

# XAC NHAN FEATURE WHITELIST

# Danh sach Feature cam truc tiep
FORBIDDEN_FEATURES = {
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp",
    "review_comment_title",
    "review_comment_message"
}

# Lay cac cot dau vao, bo order_id va nhan
feature_cols = [
    col for col in baseline.columns
    if col not in ["order_id", "is_late"]
]

print("===== XAC NHAN FEATURE WHITELIST =====")

# 1. Kiem tra du 5 Feature
assert len(whitelist) == 5
assert len(feature_cols) == 5

print("PASS: Co dung 5 Feature dau vao")

# 2. Kiem tra Feature trung voi Whitelist
assert len(set(whitelist)) == len(whitelist)
assert feature_cols == whitelist

print("PASS: Feature khop voi Whitelist")

# 3. Kiem tra Feature bi cam
leaked = set(feature_cols) & FORBIDDEN_FEATURES

assert not leaked, f"Phat hien Feature cam: {leaked}"

print("PASS: Khong chua Feature cam truc tiep")

# 4. Kiem tra cot nhan khong nam trong X
assert "is_late" not in feature_cols
assert "order_id" not in feature_cols

print("PASS: Khong dua nhan va order_id vao X")

print("\nHOAN THANH KIEM TRA FEATURE WHITELIST")


===== XAC NHAN FEATURE WHITELIST =====
PASS: Co dung 5 Feature dau vao
PASS: Feature khop voi Whitelist
PASS: Khong chua Feature cam truc tiep
PASS: Khong dua nhan va order_id vao X

HOAN THANH KIEM TRA FEATURE WHITELIST


### Kiểm tra tính toàn vẹn dữ liệu bằng SHA256

**Mục đích:** Xác minh tính toàn vẹn của dữ liệu bàn giao từ Chương 3 trước khi tiến hành chia tập và huấn luyện mô hình.

**Phương pháp:** Tính mã băm SHA256 của các file CSV đã xử lý và đối chiếu với thông tin trong `data_manifest.json`.

**Kết quả mong đợi:** Mã SHA256 của dữ liệu thực tế trùng với mã được ghi nhận trong bản bàn giao, bảo đảm dữ liệu không bị thay đổi.

In [20]:

# KIEM TRA SHA256 DU LIEU BAN GIAO

# Tim file manifest
manifest_path = next(
    (ROOT / "data").rglob("data_manifest.json"),
    None
)

if manifest_path is None:
    raise FileNotFoundError("Khong tim thay data_manifest.json")

# Doc manifest
with open(manifest_path, "r", encoding="utf-8-sig") as f:
    manifest = json.load(f)

print("===== NOI DUNG DATA MANIFEST =====")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

# Ham tinh ma SHA256
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

print("\n===== SHA256 THUC TE =====")

file_names = [
    "orders_model_ready.csv",
    "orders_baseline_input.csv",
    "orders_split_metadata.csv"
]

for name in file_names:
    path = next((ROOT / "data").rglob(name), None)

    if path is None:
        print(f"MISSING: {name}")
        continue

    actual_hash = calculate_sha256(path)

    print(f"\nFile: {name}")
    print(f"SHA256: {actual_hash}")


===== NOI DUNG DATA MANIFEST =====
{
  "notebook_version": "lam_linh_ch03_20261113",
  "source_files": [
    {
      "table": "orders",
      "file": "olist_orders_dataset.csv",
      "bytes": 17654914,
      "sha256": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf"
    },
    {
      "table": "items",
      "file": "olist_order_items_dataset.csv",
      "bytes": 15438671,
      "sha256": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279"
    },
    {
      "table": "customers",
      "file": "olist_customers_dataset.csv",
      "bytes": 9033957,
      "sha256": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2"
    },
    {
      "table": "sellers",
      "file": "olist_sellers_dataset.csv",
      "bytes": 174703,
      "sha256": "1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e"
    },
    {
      "table": "products",
      "file": "olist_products_dataset.csv",
      "bytes": 2379446,
      "sha256": "3e6569628a17fbc7

In [21]:

# XEM THONG TIN KIEM TRA SHA256

print("===== CAU TRUC DATA MANIFEST =====")
print("Cac muc trong manifest:")

for key in manifest.keys():
    print("-", key)

print("\n===== THONG TIN FILE DA XU LY =====")

target_files = [
    "orders_model_ready.csv",
    "orders_baseline_input.csv",
    "orders_split_metadata.csv"
]

for key, value in manifest.items():
    if key == "source_files":
        continue

    if isinstance(value, list):
        for item in value:
            if any(name in str(item) for name in target_files):
                print(f"\nMuc: {key}")
                print(item)

    elif isinstance(value, dict):
        for name in target_files:
            if name in str(value):
                print(f"\nMuc: {key}")
                print(json.dumps(value, indent=2, ensure_ascii=False))
                break


===== CAU TRUC DATA MANIFEST =====
Cac muc trong manifest:
- notebook_version
- source_files
- generated_at_utc
- model_file
- model_sha256
- outputs_sha256
- rows
- columns
- late_orders
- late_rate
- features
- baseline_file
- feature_policy
- label
- chronology_policy
- versions

===== THONG TIN FILE DA XU LY =====

Muc: outputs_sha256
{
  "orders_model_ready.csv": "c23d4c4d9fdcab86b73659a0273d676277a90a2dd077cbb079570d8b7673de7f",
  "orders_baseline_input.csv": "724fa2dca79eb546b7a7d37a8bb6a76e7cf3a0bfdf83493b9f295e25c2c9e7d9",
  "orders_split_metadata.csv": "51f526235df006ec010166b49ce8c0ad0e53761ef1269ae998664605766744b5",
  "feature_whitelist.json": "d50bf30ffa28e80701741e541670df84bdfca74c4d701a2cf90e43acddb6730a"
}


In [22]:

# DOI CHIEU SHA256 VOI DATA MANIFEST

print("===== XAC NHAN SHA256 DU LIEU =====")

# Lay SHA256 do Lam Linh ban giao
expected_hashes = manifest["outputs_sha256"]

results = []

for name, expected_hash in expected_hashes.items():

    # Tim file tren may
    path = next((ROOT / "data").rglob(name), None)

    if path is None:
        results.append((name, "MISSING"))
        print(f"MISSING: {name}")
        continue

    # Tinh SHA256 thuc te
    actual_hash = calculate_sha256(path)

    # So sanh
    if actual_hash == expected_hash:
        results.append((name, "PASS"))
        print(f"PASS: {name}")
    else:
        results.append((name, "FAIL"))
        print(f"FAIL: {name}")

# Kiem tra SHA256 cua model-ready
assert (
    manifest["model_sha256"]
    == expected_hashes["orders_model_ready.csv"]
), "SHA256 model-ready khong thong nhat trong manifest!"

# Ket luan
assert all(status == "PASS" for _, status in results), (
    "Du lieu ban giao khong khop SHA256!"
)

print("\nPASS: Tat ca file khop SHA256 ban giao!")


===== XAC NHAN SHA256 DU LIEU =====
PASS: orders_model_ready.csv
PASS: orders_baseline_input.csv
PASS: orders_split_metadata.csv
PASS: feature_whitelist.json

PASS: Tat ca file khop SHA256 ban giao!


### Kết luận kiểm tra dữ liệu bàn giao

Kết quả kiểm tra dữ liệu từ Chương 3 cho thấy:

- Tập dữ liệu Model-Ready gồm **96.456 dòng và 37 cột**.
- Không có mã đơn hàng `order_id` bị trùng hoặc thiếu.
- Nhãn `is_late` hợp lệ, gồm 89.922 đơn không trễ và 6.534 đơn giao trễ.
- Dữ liệu baseline sử dụng đúng **5 đặc trưng đầu vào**, phù hợp với Feature Whitelist đã được thống nhất.
- Không phát hiện các Feature bị cấm trực tiếp trong tập dữ liệu baseline.
- Mã SHA256 của 4 file đầu ra khớp hoàn toàn với `data_manifest.json`.

**Kết luận:** Dữ liệu bàn giao đáp ứng các kiểm tra về cấu trúc, nhãn, danh sách đặc trưng và tính toàn vẹn. Có thể sử dụng dữ liệu này để tiếp tục kiểm tra giao thức thực nghiệm và thực hiện Temporal Split.

Việc kiểm soát Data Leakage gián tiếp và thống nhất cách sử dụng đặc trưng giữa các mô hình vẫn cần được xác minh trong các bước tiếp theo.

## 4.1 Chia Train/Validation/Test thống nhất (Trúc Chi phụ trách)

Trúc Chi tạo split duy nhất từ `src/experiment.py` và bàn giao `split_id` cho **Ngọc Lan (RF)**, **Tấn Lên (MLP)**.

Thứ tự thực hành: Lam Linh dữ liệu → Trúc Chi split → Ngọc Lan RF và Tấn Lên MLP trên cùng Train/Validation → Trúc Chi chỉ fit Dummy, tích hợp RF/MLP → Vĩnh Chi kiểm chứng Test độc lập.

**Không huấn luyện lại RF hoặc MLP trong notebook 04; không thay đổi feature whitelist, nhãn, cutoff.**

In [23]:
from src.experiment import (get_shared_splits, NUMERIC_FEATURES, CATEGORY_FEATURES,
                            SAFE_FEATURES, choose_threshold_on_validation)
splits = get_shared_splits(PROCESSED)
train, val, test = splits["train"], splits["val"], splits["test"]
split_info = splits["metadata"]
split_id = split_info["split_id"]
FEATURES = list(SAFE_FEATURES)
print("Shared split id:", split_id)
display(pd.DataFrame([
    {"split":label, "n":len(splits[label]),
     "late_rate":splits[label]["is_late"].mean()}
    for label in ("train","val","test")
]))
print("Ngọc Lan (RF), Tấn Lên (MLP), Trúc Chi (Dummy) dùng cùng split_id.")

Shared split id: 72c64fdd7b6de725a531


,split,n,late_rate
0,train,64416,0.068120
1,val,12551,0.042945
2,test,14469,0.042781


Ngọc Lan (RF), Tấn Lên (MLP), Trúc Chi (Dummy) dùng cùng split_id.


### 4.1.1. Kiểm tra kết quả Temporal Split

**Mục đích:** Kiểm tra tính hợp lệ của ba tập Train, Validation và Test sau khi chia dữ liệu theo thời gian.

**Nội dung kiểm tra:**
- Thống kê số lượng đơn hàng trong từng tập dữ liệu.
- Kiểm tra tính duy nhất của `order_id`, đảm bảo không có đơn hàng trùng lặp giữa các tập.
- Xác nhận thứ tự thời gian Train → Validation → Test dựa trên `order_approved_at`.
- Kiểm tra các tập sử dụng thống nhất 5 đặc trưng đầu vào.
- Đối chiếu số lượng đơn hàng trước và sau khi chia dữ liệu.

**Kết quả mong đợi:** Ba tập dữ liệu không trùng đơn hàng, tuân thủ thứ tự thời gian và sử dụng cùng Feature Whitelist.

In [24]:

# 4.1.1 - KIEM TRA KET QUA TEMPORAL SPLIT

print("===== KIEM TRA TEMPORAL SPLIT =====")

# 1. Kiem tra so luong du lieu
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

# 2. Kiem tra order_id trung lap
train_ids = set(train["order_id"])
val_ids = set(val["order_id"])
test_ids = set(test["order_id"])

assert train["order_id"].notna().all()
assert val["order_id"].notna().all()
assert test["order_id"].notna().all()

assert train["order_id"].is_unique
assert val["order_id"].is_unique
assert test["order_id"].is_unique

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nPASS: Khong trung order_id giua 3 tap")

# 3. Kiem tra thu tu thoi gian
time_col = "order_approved_at"

train_time = pd.to_datetime(train[time_col])
val_time = pd.to_datetime(val[time_col])
test_time = pd.to_datetime(test[time_col])

assert train_time.notna().all()
assert val_time.notna().all()
assert test_time.notna().all()

assert train_time.max() <= val_time.min()
assert val_time.max() <= test_time.min()

print("PASS: Train -> Validation -> Test theo thoi gian")

# 4. Doc Feature Whitelist tu file JSON
whitelist_path = next(
    (ROOT / "data").rglob("feature_whitelist.json"),
    None
)

if whitelist_path is None:
    raise FileNotFoundError(
        "Khong tim thay feature_whitelist.json"
    )

with open(whitelist_path, "r", encoding="utf-8-sig") as f:
    whitelist = json.load(f)

# 5. Kiem tra 5 Feature dung chung
assert isinstance(whitelist, list)
assert len(FEATURES) == 5
assert FEATURES == whitelist

for data in [train, val, test]:
    assert all(col in data.columns for col in FEATURES)

print("PASS: Ca 3 tap su dung cung 5 Feature")

# 6. Doi chieu so luong du lieu
model_ready_path = (
    ROOT / "data" / "processed" / "orders_model_ready.csv"
)

model_ready_ids = pd.read_csv(
    model_ready_path,
    usecols=["order_id"]
)["order_id"]

all_split_ids = train_ids | val_ids | test_ids

assert all_split_ids.issubset(set(model_ready_ids))

total = len(train) + len(val) + len(test)
excluded = len(model_ready_ids) - total

assert excluded >= 0

print("\n===== THONG KE DU LIEU =====")
print("Model-Ready:", len(model_ready_ids))
print("Sau khi chia:", total)
print("Khong thuoc 3 tap:", excluded)

# 7. Kiem tra Split ID
assert split_id == split_info["split_id"]

print("\nShared Split ID:", split_id)

print("\nPASS: Kiem tra 4.1.1 hoan thanh")


===== KIEM TRA TEMPORAL SPLIT =====
Train: 64416
Validation: 12551
Test: 14469

PASS: Khong trung order_id giua 3 tap
PASS: Train -> Validation -> Test theo thoi gian
PASS: Ca 3 tap su dung cung 5 Feature

===== THONG KE DU LIEU =====
Model-Ready: 96456
Sau khi chia: 91436
Khong thuoc 3 tap: 5020

Shared Split ID: 72c64fdd7b6de725a531

PASS: Kiem tra 4.1.1 hoan thanh


### 4.1.2. Kiểm tra Cutoff và Label Maturity

**Mục đích:** Xác minh các mốc thời gian chia dữ liệu và đảm bảo mô hình không sử dụng kết quả giao hàng chưa xảy ra tại thời điểm huấn luyện hoặc đánh giá.

**Phương pháp thực hiện:**
- Xác định `validation_start` và `test_start` từ Split Metadata.
- Kiểm tra ngày giao thực tế của Train phải trước thời điểm bắt đầu Validation.
- Kiểm tra ngày giao thực tế của Validation phải trước thời điểm bắt đầu Test.
- Thống kê số đơn hàng bị loại do chưa đủ thời gian quan sát nhãn (Label Maturity).

**Kết quả mong đợi:** Các tập dữ liệu tuân thủ quy tắc Cutoff và Label Maturity, không sử dụng thông tin giao hàng từ tương lai.

In [25]:

# 4.1.2 - KIEM TRA CUTOFF VA LABEL MATURITY

print("===== KIEM TRA CUTOFF =====")

validation_start = pd.to_datetime(
    split_info["validation_start"]
)
test_start = pd.to_datetime(
    split_info["test_start"]
)

print("Validation start:", validation_start)
print("Test start:", test_start)

assert validation_start < test_start

# Kiem tra Label Maturity
train_delivery = pd.to_datetime(
    train["order_delivered_customer_date"]
)
val_delivery = pd.to_datetime(
    val["order_delivered_customer_date"]
)

assert train_delivery.notna().all()
assert val_delivery.notna().all()

assert (train_delivery < validation_start).all()
assert (val_delivery < test_start).all()

print("\nPASS: Train khong dung nhan tuong lai")
print("PASS: Validation khong dung nhan tuong lai")

# Thong ke cac don bi loai
excluded_train = split_info["excluded_train_unmatured"]
excluded_val = split_info["excluded_validation_unmatured"]

total_excluded = excluded_train + excluded_val

print("\n===== LABEL MATURITY =====")
print("Train bi loai:", excluded_train)
print("Validation bi loai:", excluded_val)
print("Tong so don bi loai:", total_excluded)

# Doi chieu so luong
model_ready_n = len(
    pd.read_csv(
        PROCESSED / "orders_model_ready.csv",
        usecols=["order_id"]
    )
)

assert (
    len(train) + len(val) + len(test) + total_excluded
    == model_ready_n
)

assert split_info["n_train"] == len(train)
assert split_info["n_validation"] == len(val)
assert split_info["n_test"] == len(test)

print("\nPASS: So luong du lieu khop metadata")
print("PASS: Kiem tra 4.1.2 hoan thanh")


===== KIEM TRA CUTOFF =====
Validation start: 2018-04-16 09:31:24
Test start: 2018-06-21 12:41:03

PASS: Train khong dung nhan tuong lai
PASS: Validation khong dung nhan tuong lai

===== LABEL MATURITY =====
Train bi loai: 3103
Validation bi loai: 1917
Tong so don bi loai: 5020

PASS: So luong du lieu khop metadata
PASS: Kiem tra 4.1.2 hoan thanh


### 4.1.3. Xác nhận Split ID và bàn giao cho nhóm

**Mục đích:** Xác nhận kết quả chia dữ liệu thống nhất và cung cấp thông tin Split cho các thành viên phụ trách huấn luyện mô hình.

**Nội dung thực hiện:**
- Kiểm tra `split_id` và mã SHA256 của dữ liệu gốc đã xử lý.
- Xác nhận số lượng Train, Validation, Test và danh sách 5 đặc trưng đầu vào.
- Lưu thông tin chia dữ liệu để phục vụ việc đối chiếu kết quả thực nghiệm.
- Bàn giao `split_id` cho Ngọc Lan (Random Forest) và Tấn Lên (MLP).

**Kết quả mong đợi:** Các mô hình Dummy Classifier, Random Forest và MLP sử dụng cùng quy tắc chia dữ liệu, Feature Whitelist và phiên bản dữ liệu đã được xác nhận.

In [26]:

# 4.1.3 - XAC NHAN SPLIT ID VA BAN GIAO

# Ham tinh SHA256
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


print("===== XAC NHAN SHARED SPLIT =====")

# 1. Kiem tra Split ID
assert split_id == split_info["split_id"]
print("PASS: Split ID hop le")

# 2. Kiem tra SHA256 cua Model-Ready
model_path = PROCESSED / "orders_model_ready.csv"

assert model_path.exists(), "Khong tim thay Model-Ready"

actual_sha256 = calculate_sha256(model_path)

assert actual_sha256 == split_info["csv_sha256"], (
    "SHA256 khong khop du lieu Model-Ready!"
)

print("PASS: SHA256 khop du lieu Model-Ready")

# 3. Kiem tra Feature Whitelist
whitelist_path = next(
    (ROOT / "data").rglob("feature_whitelist.json"),
    None
)

if whitelist_path is None:
    raise FileNotFoundError("Khong tim thay Feature Whitelist")

with open(whitelist_path, "r", encoding="utf-8-sig") as f:
    whitelist = json.load(f)

assert FEATURES == whitelist
assert len(FEATURES) == 5

print("PASS: Feature Whitelist thong nhat")

# 4. Kiem tra so luong theo Metadata
assert split_info["n_train"] == len(train)
assert split_info["n_validation"] == len(val)
assert split_info["n_test"] == len(test)

print("PASS: So luong 3 tap khop Metadata")

# 5. Tao thong tin ban giao
handoff = {
    "split_id": split_id,
    "dataset_sha256": actual_sha256,
    "features": FEATURES,
    "validation_start": str(split_info["validation_start"]),
    "test_start": str(split_info["test_start"]),
    "n_train": len(train),
    "n_validation": len(val),
    "n_test": len(test),
    "train_late_rate": float(train["is_late"].mean()),
    "val_late_rate": float(val["is_late"].mean()),
    "test_late_rate": float(test["is_late"].mean()),
    "excluded_train": split_info["excluded_train_unmatured"],
    "excluded_validation": split_info["excluded_validation_unmatured"]
}

# 6. Luu file ban giao
output_dir = ROOT / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "t12_split_handoff.json"

output_file.write_text(
    json.dumps(handoff, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 7. Hien thi ket qua
print("\n===== THONG TIN BAN GIAO =====")
print("Shared Split ID:", split_id)
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))
print("So Feature:", len(FEATURES))
print("File ban giao:", output_file)

print("\nPASS: Hoan thanh 4.1.3")


===== XAC NHAN SHARED SPLIT =====
PASS: Split ID hop le
PASS: SHA256 khop du lieu Model-Ready
PASS: Feature Whitelist thong nhat
PASS: So luong 3 tap khop Metadata

===== THONG TIN BAN GIAO =====
Shared Split ID: 72c64fdd7b6de725a531
Train: 64416
Validation: 12551
Test: 14469
So Feature: 5
File ban giao: d:\ĐAKHDL\Do_An_Khoa_hoc_du_lieu---GR02\reports\tables\t12_split_handoff.json

PASS: Hoan thanh 4.1.3


### Kết luận mục 4.1

Quá trình chia dữ liệu theo thời gian (Temporal Split) đã tạo ra ba tập:

- **Train:** 64.416 đơn hàng.
- **Validation:** 12.551 đơn hàng.
- **Test:** 14.469 đơn hàng.

Có 5.020 đơn hàng bị loại theo quy tắc Label Maturity, gồm 3.103 đơn thuộc Train và 1.917 đơn thuộc Validation.

Kết quả kiểm tra xác nhận các tập dữ liệu không trùng lặp `order_id`, tuân thủ thứ tự thời gian, sử dụng chung 5 đặc trưng hợp lệ và có mã SHA256 khớp với dữ liệu bàn giao.

**Shared Split ID:** `72c64fdd7b6de725a531`

Thông tin chia tập đã được lưu trong `t12_split_handoff.json`, làm cơ sở để các mô hình Dummy Classifier, Random Forest và MLP sử dụng thống nhất trong quá trình thực nghiệm.

## 4.2 Tiền xử lý và huấn luyện Dummy Classifier – Trúc Chi (TV4)

Trúc Chi chỉ fit **Dummy Classifier**, tạo baseline trên Train chung và tính xác suất Validation. Preprocessing phải fit Train-only.

RF do Ngọc Lan đã train; MLP do Tấn Lên đã train. Trúc Chi tiếp nhận các artifact, kiểm tra metadata và tích hợp thay vì train thêm.

In [ ]:
import json
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

FEATURES = list(SAFE_FEATURES)
X_train, y_train = train[FEATURES], train["is_late"].astype(int)
X_val, y_val = val[FEATURES], val["is_late"].astype(int)
X_test, y_test = test[FEATURES], test["is_late"].astype(int)
assert y_train.nunique()==2 and y_val.nunique()==2
dummy_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), CATEGORY_FEATURES),
])
dummy = Pipeline([
    ("preprocess", dummy_prep),
    ("dummy", DummyClassifier(strategy="prior")),
])
dummy.fit(X_train, y_train)  # Trúc Chi CHỈ fit Dummy
p_dummy_val = dummy.predict_proba(X_val)[:,1]
print("TV4 đã huấn luyện Dummy; RF và MLP nhận artifact từ Ngọc Lan và Tấn Lên.")

## 4.3 Tiếp nhận Random Forest của Ngọc Lan; 4.4 tiếp nhận MLP của Tấn Lên

**4.3 Random Forest:** Ngọc Lan huấn luyện tại notebook 02, Trúc Chi nhận `ngoc_lan_random_forest.joblib`, `rf_val_predictions.csv`, `rf_metadata.json`.

**4.4 MLP:** Tấn Lên trực tiếp huấn luyện tại notebook 01 và bàn giao `tan_len_mlp.joblib`, `mlp_val_predictions.csv`, `mlp_metadata.json`. Trúc Chi vẫn là người phụ trách biên soạn mục 4.4 trong **Chương 4**.

Kiểm tra bắt buộc: `split_id`, dataset SHA256, feature whitelist, Validation order_id/y_true, xác suất `predict_proba` phải trùng file bàn giao. **Không được gọi `fit` với RF hay MLP.**

In [ ]:
rf_path = ROOT / "models" / "ngoc_lan_random_forest.joblib"
mlp_path = ROOT / "models" / "tan_len_mlp.joblib"
rf_meta_path = PROCESSED / "rf_metadata.json"
mlp_meta_path = PROCESSED / "mlp_metadata.json"
rf_val_path = PROCESSED / "rf_val_predictions.csv"
mlp_val_path = PROCESSED / "mlp_val_predictions.csv"
required = (rf_path, mlp_path, rf_meta_path, mlp_meta_path, rf_val_path, mlp_val_path)
missing = [p.name for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Chưa nhận RF/MLP từ Ngọc Lan/Tấn Lên: " + ", ".join(missing))
rf_meta = json.loads(rf_meta_path.read_text(encoding="utf-8"))
mlp_meta = json.loads(mlp_meta_path.read_text(encoding="utf-8"))
rf_val_file = pd.read_csv(rf_val_path)
mlp_val_file = pd.read_csv(mlp_val_path)
for name, meta, owner, validation_file, prob_col in [
    ("rf", rf_meta, "Ngọc Lan", rf_val_file, "p_rf"),
    ("mlp", mlp_meta, "Tấn Lên", mlp_val_file, "p_mlp"),
]:
    assert meta["model"] == name and meta["owner"] == owner
    assert meta["split_id"] == split_id
    assert meta["dataset_sha256"] == split_info["csv_sha256"]
    assert meta["features"] == FEATURES
    assert meta["validation_n"] == len(val)
    assert np.array_equal(validation_file["order_id"].to_numpy(), val["order_id"].to_numpy())
    assert np.array_equal(validation_file["y_true"].to_numpy(), y_val.to_numpy())
rf = joblib.load(rf_path)
mlp = joblib.load(mlp_path)
p_rf = rf.predict_proba(X_val)[:,1]
p_mlp = mlp.predict_proba(X_val)[:,1]
assert np.allclose(p_rf, rf_val_file["p_rf"].to_numpy(), atol=1e-7)
assert np.allclose(p_mlp, mlp_val_file["p_mlp"].to_numpy(), atol=1e-7)
models = {"dummy": dummy, "rf": rf, "mlp": mlp}
validation_prob = {"dummy": p_dummy_val, "rf": p_rf, "mlp": p_mlp}
validation_results = [
    {"model": name,
     "owner": {"rf":"Ngọc Lan", "mlp":"Tấn Lên", "dummy":"Trúc Chi"}[name],
     "val_AP": float(average_precision_score(y_val, validation_prob[name]))}
    for name in ("dummy", "rf", "mlp")
]
display(pd.DataFrame(validation_results).sort_values("val_AP",ascending=False))
print("TV4 chỉ fit Dummy, RF/MLP đã được kiểm chứng mà không fit lại.")

## 4.5 Tổng hợp và chọn champion/threshold chỉ trên Validation

- Random Forest: lấy threshold do Ngọc Lan chốt từ Validation.
- MLP: lấy threshold do Tấn Lên chốt từ Validation.
- Dummy: Trúc Chi chọn threshold theo F1 Validation.
- Trúc Chi so sánh AP Validation của RF/MLP để chọn champion. Không dùng Test quyết định bất kỳ tham số nào.

In [ ]:
from src.experiment import classification_metrics
thresholds = {
    "rf": float(rf_meta["validation_metrics"]["threshold"]),
    "mlp": float(mlp_meta["validation_metrics"]["threshold"]),
}
dummy_val_metrics = choose_threshold_on_validation(y_val, validation_prob["dummy"])
thresholds["dummy"] = float(dummy_val_metrics["threshold"])
val_metric_rows = []
owners = {"rf":"Ngọc Lan","mlp":"Tấn Lên","dummy":"Trúc Chi"}
for name in ("dummy","rf","mlp"):
    result = classification_metrics(y_val, validation_prob[name], thresholds[name])
    val_metric_rows.append({"model":name, "owner":owners[name],
                            **result, "dataset":"Validation"})
for name, meta in (("rf",rf_meta),("mlp",mlp_meta)):
    actual_ap = next(row["AP"] for row in val_metric_rows if row["model"]==name)
    assert np.isclose(actual_ap, float(meta["validation_metrics"]["AP"]), atol=1e-8)
display(pd.DataFrame(val_metric_rows))
champion = max((r for r in validation_results if r["model"] in ("rf","mlp")),
               key=lambda r:r["val_AP"])["model"]
chosen_threshold = thresholds[champion]
print("Champion dựa trên AP Validation:", champion, "threshold:", chosen_threshold)

## 4.6 Đóng băng dự đoán Test và bàn giao Vĩnh Chi

Chỉ sau khi nhận RF của Ngọc Lan, MLP của Tấn Lên và huấn luyện Dummy, Trúc Chi mới xuất Test predictions.

Lưu `test_predictions.csv` (ba cột xác suất) + `experiment_metadata.json` ghi ba model owners và `split_id`, `models/dummy_truc_chi.joblib`, `models/champion.joblib`. **Không tạo file MLP mới; dùng artifact Tấn Lên đã bàn giao.**

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
test_probabilities = {
    name: models[name].predict_proba(X_test)[:,1]
    for name in ("dummy","rf","mlp")
}
predictions = pd.DataFrame({
    "order_id":test["order_id"].to_numpy(),
    "y_true":y_test.to_numpy(),
    "p_dummy":test_probabilities["dummy"],
    "p_rf":test_probabilities["rf"],
    "p_mlp":test_probabilities["mlp"],
})
predictions["y_pred_champion"] = (
    predictions["p_"+champion].to_numpy() >= chosen_threshold
).astype(int)
assert predictions["order_id"].is_unique and len(predictions)==len(test)
predictions.to_csv(PROCESSED / "test_predictions.csv", index=False)
joblib.dump(dummy, MODEL_DIR / "dummy_truc_chi.joblib")
joblib.dump(models[champion], MODEL_DIR / "champion.joblib")
meta = {
    "prediction_time":"order_approved_at", "features":FEATURES,
    "split_id":split_id, "split_metadata":split_info,
    "model_owners":{"rf":"Ngọc Lan","dummy":"Trúc Chi","mlp":"Tấn Lên"},
    "rf_source":"models/ngoc_lan_random_forest.joblib",
    "mlp_source":"models/tan_len_mlp.joblib",
    "champion":champion, "thresholds_from_val":thresholds,
    "chosen_threshold":chosen_threshold,
    "validation_AP":{r["model"]:r["val_AP"] for r in validation_results},
    "test_n":len(test), "random_state":42,
    "label":"DATE(actual)>DATE(estimated) on delivered labeled orders",
}
(PROCESSED / "experiment_metadata.json").write_text(
    json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8"
)
print("Trúc Chi đã tích hợp RF Ngọc Lan + MLP Tấn Lên + Dummy TV4; frozen Test cho Vĩnh Chi.")

## 4.7 Checklist nghiệm thu Chương 4
- [ ] Split Train/Validation/Test chung cho RF, MLP, Dummy có split_id, cutoff, dữ liệu SHA256.
- [ ] Trúc Chi CHỈ gọi fit Dummy, không gọi fit RF hoặc MLP.
- [ ] Đã nhận RF artifact/metadata/Validation của Ngọc Lan.
- [ ] Đã nhận MLP artifact/metadata/Validation của Tấn Lên (notebook 01).
- [ ] 4.4 trình bày kiến trúc, cấu hình, training loss/Validation của TV1, không nhận công train của TV4.
- [ ] Đã kiểm tra order_id, y_true, Validation p_rf và p_mlp trước khi tích hợp.
- [ ] Champion và thresholds chốt từ Validation; exported frozen Test, metadata và Dummy fitted artifact.
- [ ] Vĩnh Chi nhận Test và toàn bộ fitted model files, không tuning lại Test.